# Visual token pruning: CPU-vs-GPU sweep (Kaggle GPU)

Companion run to the CPU (Apple M4 Pro) and x86 (GitHub Actions) sweeps
already recorded in `NOTES.md`/`vtp-cpu-plan-v2.md`. This notebook builds
`waleedabujaish/llama.cpp` with CUDA, re-validates correctness on this
backend (Gate 1), re-runs the same keep-ratio latency matrix, and runs a
TextVQA accuracy-vs-keep-ratio sweep on the identical quantized build used
on CPU/x86 (`second-state/Llava-v1.5-7B-GGUF`, Q4_K_M + F16 mmproj) — same
model, same manifest, only the backend differs, so speed and accuracy
numbers are directly comparable across the three platforms.

**Before running:** in Kaggle's notebook settings (right sidebar), turn on
both **Accelerator: GPU T4 x2 or P100** and **Internet: on** (needed to
clone the repos, download the model, and stream TextVQA images). Then
**Run All**.

**What this notebook does NOT do by default:** MME/POPE accuracy evals —
see the disabled stub section near the end for why, and what's needed to
turn it on. Everything else in the task list runs by default.

**Honesty note on things I could not test.** I have no GPU in my own
environment, so nothing past "does `nvcc`/`nvidia-smi` exist" in this
notebook has been run end-to-end before you run it on Kaggle. Every step
that has a real chance of silently doing the wrong thing instead of
failing (GPU offload not actually happening, a CUDA build being
non-deterministic between two encodes, the accuracy phase's server mode
producing different text than the CLI path) has an explicit, loud check
built in — see the markdown notes above each such cell. Read the printed
output on your first run, don't just trust "Run All" finished with no
red cells.


## 1. Config

Edit these before running if you want to change scope. Defaults target
the full task as specified: all 6 latency ratios, all 200 pinned TextVQA
samples across all 6 ratios.


In [ ]:
# ---- pinned commits (same discipline as .github/workflows/x86-cpu-sweep.yml) ----
VISION_TOKENS_SHA = "b48b90b15b9c55cc73a4d921a52dd9570cefaa8d"       # this repo -- MUST be at or after this commit (adds GPU support to the scripts below)
LOCAL_TEST_BOTH_SHA = "5b90586353cadd295e46c23118c1329cfdc86d3c"     # llama.cpp: fixes only, no pruning code (Gate 1 reference)
VISUAL_TOKEN_PRUNING_SHA = "82f2ccb5cc4c12f610726c71a6f941c64e11daac"  # llama.cpp: fixes + --visual-keep

# ---- model checksums (second-state/Llava-v1.5-7B-GGUF, same as CPU/x86 runs) ----
MODEL_GGUF_SHA256 = "2687b20ac8b7a23f6c70296d5b1e7f908fef2ce4769ecdebd1bb9503528a75bf"
MMPROJ_SHA256 = "50da4e5b0a011615f77686f9b02613571e65d23083c225e107c08c3b1775d9b1"

# ---- latency sweep ----
LATENCY_RATIOS = "1.0,0.75,0.5,0.25,0.1,0.05"   # same 6 cells as CPU/x86; sub-0.05 H2-chasing ratios intentionally excluded (already answered as ambiguous on CPU)
LATENCY_RUNS = 5
LATENCY_WARMUP = 1
# CPU's 30s cooldown was specifically a fix for sustained -t8 thermal throttling on a
# fanless/limited-cooling laptop chip (see NOTES.md "Thermal drift"). Datacenter T4/P100
# have real cooling and are built for sustained load, so that specific failure mode is not
# expected to reproduce here -- but this is a judgment call, not something I measured (no
# GPU available to me). A short cooldown is kept anyway, mainly to let the process fully
# exit and release the CUDA context/VRAM between invocations rather than for thermal
# recovery. If the printed per-run timings drift upward over a cell the way the CPU ones
# did, that's a sign this guess was wrong -- increase it and re-run (already-done cells
# are skipped, so this is cheap to correct).
LATENCY_COOLDOWN_S = 5

# ---- accuracy sweep ----
# The pinned manifest (assets/phase1/textvqa200_manifest.jsonl) has exactly 200 samples.
# Expanding beyond 200 would mean drawing a new, unpinned slice of TextVQA -- out of
# scope here (would need its own pin + provenance first, like the 200 did). "Expand if
# time allows" is available via ACCURACY_RATIOS / the MME-POPE section below instead.
ACCURACY_LIMIT = 0          # 0 = full 200 pinned samples
ACCURACY_RATIOS = "1.0,0.75,0.5,0.25,0.1,0.05"
ACCURACY_MODE = "auto"      # auto = try server mode (fast), verify against CLI, fall back to CLI (slow, proven) if they disagree -- see section 15

import os
os.environ["VISION_TOKENS_SHA"] = VISION_TOKENS_SHA
print("config loaded")


## 2. GPU / CUDA detection

Fail loudly here rather than silently building a CPU-only binary that
happens to still run (the whole point of this notebook is the GPU
numbers). If this cell raises, stop and check the Kaggle accelerator
setting before doing anything else.


In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr

def nvidia_smi_field(field):
    rc, out, err = sh(f"nvidia-smi --query-gpu={field} --format=csv,noheader")
    return out.strip().splitlines()[0].strip() if rc == 0 and out.strip() else ""

# Presence is gated on `name` alone, not a combined query with the other fields: an
# older nvidia-smi/driver that doesn't recognize one field name (compute_cap is a
# relatively recent addition) fails the WHOLE combined query, which would misreport
# "no GPU" instead of just "one field unavailable" -- a wasted-hour kind of false
# alarm if it happened here. The richer fields below are best-effort.
gpu_name = nvidia_smi_field("name")
if not gpu_name:
    raise RuntimeError(
        "nvidia-smi did not report a GPU name.\n"
        "This notebook requires a GPU accelerator. In Kaggle's notebook settings "
        "(right sidebar -> Accelerator), select a GPU (T4 x2 or P100) and re-run. "
        "Refusing to silently continue on a CPU-only session."
    )
gpu_mem = nvidia_smi_field("memory.total")
gpu_driver = nvidia_smi_field("driver_version")
gpu_cc = nvidia_smi_field("compute_cap")
print(f"GPU detected: {gpu_name} | {gpu_mem} | driver {gpu_driver} | compute capability {gpu_cc}")

rc, out, err = sh("nvcc --version")
if rc != 0:
    raise RuntimeError(
        "nvcc not found (rc={}). Kaggle's GPU images are expected to ship the CUDA "
        "toolkit preinstalled; its absence means -DGGML_CUDA=ON will fail to build "
        "or silently produce a CPU-only binary. Refusing to continue.".format(rc)
    )
print(out.strip().splitlines()[-1])

# sanitize GPU name into a platform tag, e.g. "Tesla T4" -> "kaggle-gpu-tesla-t4"
PLATFORM_TAG = "kaggle-gpu-" + gpu_name.lower().replace(" ", "-")
print(f"platform_tag: {PLATFORM_TAG}")


## 3. Shared helpers

Every meaningful step below goes through `run()`/`run_capture()`, which
raise on a non-zero exit. Plain `!shell` one-liners don't stop "Run All"
on failure the way a raised Python exception does — for a notebook meant
to run unattended, a silent continue past a failed step is worse than a
stopped one.


In [ ]:
import subprocess, sys, time, json, os, glob, hashlib, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

def run(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(str(c) for c in cmd)}" + (f"   (cwd={cwd})" if cwd else ""), flush=True)
    full_env = {**os.environ, **(env or {})}
    result = subprocess.run(cmd, cwd=cwd, env=full_env)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed (exit {result.returncode}): {' '.join(str(c) for c in cmd)}")
    return result.returncode

def run_capture(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(str(c) for c in cmd)}" + (f"   (cwd={cwd})" if cwd else ""), flush=True)
    full_env = {**os.environ, **(env or {})}
    result = subprocess.run(cmd, cwd=cwd, env=full_env, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed (exit {result.returncode}): {' '.join(str(c) for c in cmd)}")
    return result

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()

print("helpers loaded")


## 4. Clone repos and verify pinned commits

Same discipline as `.github/workflows/x86-cpu-sweep.yml`'s "Verify pinned
commits" step: checkout by branch name, then hard-assert `git rev-parse
HEAD` matches the recorded SHA before building anything, rather than
trusting the branch still points where it did when this notebook was
written.


In [ ]:
# vision_tokens: full clone (no --depth), and its pin is checked as "is this commit an
# ancestor of HEAD", not exact equality. Reason: committing an update to VISION_TOKENS_SHA
# necessarily creates a new commit whose hash isn't known until after it's made, so this
# repo's own pin can never exactly equal live HEAD by the time anyone clones it -- unlike
# the llama.cpp pins below, which really do need to be exact (Gate 1's reference build
# must be precisely those commits, not "at least these or later"). Ancestor-checking needs
# real history, hence no --depth 1 here; this repo is small and young, so that's cheap.
if WORK.joinpath("vision_tokens").exists():
    print("vision_tokens already cloned, skipping")
else:
    run(["git", "clone", "https://github.com/waleedabujaish/llama.cpp-visual-token-pruning", "vision_tokens"], cwd=WORK)
vt_head = run_capture(["git", "rev-parse", "HEAD"], cwd=WORK / "vision_tokens").stdout.strip()
vt_pin_ok = run_capture(["git", "merge-base", "--is-ancestor", VISION_TOKENS_SHA, "HEAD"],
                         cwd=WORK / "vision_tokens", check=False).returncode == 0
if not vt_pin_ok:
    raise RuntimeError(f"vision_tokens HEAD ({vt_head}) does not include pinned commit "
                        f"{VISION_TOKENS_SHA} as an ancestor -- either the pin is wrong/from a "
                        f"different branch, or something is misconfigured.")
print(f"OK: vision_tokens HEAD ({vt_head}) includes pinned commit {VISION_TOKENS_SHA}")

# --depth 1: only HEAD of the pinned branch is needed (rev-parse HEAD below still
# gives an accurate SHA for a shallow clone -- this doesn't weaken the pin check,
# it just skips downloading llama.cpp's multi-year history). These two DO need exact
# equality, not ancestry -- see the comment above.
if WORK.joinpath("llama.cpp-pristine").exists():
    print("llama.cpp-pristine already cloned, skipping")
else:
    run(["git", "clone", "--depth", "1", "--branch", "local-test-both", "https://github.com/waleedabujaish/llama.cpp", "llama.cpp-pristine"], cwd=WORK)
pristine_head = run_capture(["git", "rev-parse", "HEAD"], cwd=WORK / "llama.cpp-pristine").stdout.strip()
if pristine_head != LOCAL_TEST_BOTH_SHA:
    raise RuntimeError(f"local-test-both HEAD is {pristine_head}, expected {LOCAL_TEST_BOTH_SHA}")
print(f"OK: local-test-both HEAD matches pinned {pristine_head}")

if WORK.joinpath("llama.cpp-prune").exists():
    print("llama.cpp-prune already cloned, skipping")
else:
    run(["git", "clone", "--depth", "1", "--branch", "visual-token-pruning", "https://github.com/waleedabujaish/llama.cpp", "llama.cpp-prune"], cwd=WORK)
prune_head = run_capture(["git", "rev-parse", "HEAD"], cwd=WORK / "llama.cpp-prune").stdout.strip()
if prune_head != VISUAL_TOKEN_PRUNING_SHA:
    raise RuntimeError(f"visual-token-pruning HEAD is {prune_head}, expected {VISUAL_TOKEN_PRUNING_SHA}")
print(f"OK: visual-token-pruning HEAD matches pinned {prune_head}")


## 5. Python dependencies

`numpy`/`pillow` for the gate checks and image handling; `datasets` to
stream TextVQA (materializing the 200 pinned images). No `requests` --
the accuracy sweep's server mode uses stdlib `urllib`, deliberately, to
avoid one more thing that could be missing.


In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pillow", "datasets"])
print("python deps installed")


## 6. Build both binaries with CUDA

`-DGGML_CUDA=ON`, otherwise Release defaults. Building `llama-mtmd-cli`
(latency + Gate 1 + CLI-mode accuracy) and `llama-server` (accuracy
sweep's server mode, see section 15) for the prune build; pristine only
needs `llama-mtmd-cli` (Gate 1 reference, no server mode needed there).


### 6a. Work around a known container gotcha: `libcuda.so` (unversioned) missing

`nvidia-smi`/`nvcc` above prove the driver and toolkit both work, but
CMake's `FindCUDAToolkit` builds the `CUDA::cuda_driver` target (which
`ggml-cuda` links against) via a `find_library` search for `libcuda.so`
(unversioned) — first in the toolkit's own directories (including a
`stubs/` subdirectory NVIDIA's toolkit packages normally ship a link-only
stub in), then falling back to nothing. Two independent things can break
this on a container image: the toolkit's own `stubs/` directory may not
be present at all on a non-devel image, and/or the *real* driver library
(injected by nvidia-container-toolkit, not apt-installed) may live
somewhere this search doesn't check — commonly `/usr/local/nvidia/lib64`,
not the apt-package path `/usr/lib/x86_64-linux-gnu` a first attempt at
this fix assumed. Search broadly, report exactly what was found and
where, and don't silently no-op if nothing turns up (the build cell below
has a verified fallback -- `-DGGML_CUDA_NO_VMM=ON` -- for exactly that
case, so a diagnostic miss here doesn't have to mean stuck).


In [ ]:
CUDA_STUB_DIR = None  # set below if a fallback symlink location was needed
CUDA_TOOLKIT_ROOT = "/usr/local/cuda"

def find_libcuda_unrestricted(timeout_s=45):
    """Slow, last-resort fallback -- only called if the fast targeted glob search
    below finds nothing. Has a hard timeout: an unrestricted `find /` occasionally
    hangs a long time on some filesystems (observed locally on macOS; unconfirmed
    whether Kaggle's container has anything similar, but there's no reason to risk
    stalling a live session on a search that's a fallback in the first place)."""
    try:
        r = subprocess.run(["bash", "-c", "find / -xdev -name 'libcuda.so*' 2>/dev/null"],
                            capture_output=True, text=True, timeout=timeout_s)
        return r.stdout
    except subprocess.TimeoutExpired:
        print(f"(unrestricted filesystem search timed out after {timeout_s}s, skipping -- "
              f"not fatal, just means less diagnostic detail if the targeted search below "
              f"also comes up empty)")
        return ""

def ensure_libcuda_so():
    global CUDA_STUB_DIR
    # Broad, fast, targeted search first: apt-installed paths, the CUDA toolkit's own
    # stub locations, AND the nvidia-container-toolkit bind-mount convention
    # (/usr/local/nvidia/lib64) -- confirmed (via research into Kaggle's own public
    # Dockerfile) to be where the REAL driver library actually lives on Kaggle's GPU
    # image, listed first in its LD_LIBRARY_PATH. A prior fix attempt only checked the
    # apt-package path and found nothing there, which is why it was a silent no-op.
    search_globs = [
        "/usr/local/nvidia/lib64/libcuda.so*",
        "/usr/local/nvidia/lib/libcuda.so*",
        "/usr/lib/x86_64-linux-gnu/libcuda.so*",
        "/usr/lib/x86_64-linux-gnu/nvidia/current/libcuda.so*",
        f"{CUDA_TOOLKIT_ROOT}/compat/libcuda.so*",
        f"{CUDA_TOOLKIT_ROOT}/lib64/stubs/libcuda.so*",
        f"{CUDA_TOOLKIT_ROOT}/targets/*/lib/stubs/libcuda.so*",
        "/usr/lib/wsl/lib/libcuda.so*",
    ]
    candidates = sorted({p for pat in search_globs for p in glob.glob(pat)})
    print("libcuda.so* candidates (targeted search):", candidates or "(none)")
    if not candidates:
        print("targeted search found nothing -- falling back to an unrestricted filesystem search ...")
        unrestricted = find_libcuda_unrestricted()
        print("libcuda.so* found (unrestricted search):\n" + (unrestricted or "  (none)"))
        candidates = sorted(set(unrestricted.strip().splitlines())) if unrestricted.strip() else []

    if any(c.endswith("libcuda.so") for c in candidates):
        print("OK: libcuda.so (unversioned) already present, no fix needed")
        return
    versioned = [c for c in candidates if not c.endswith("libcuda.so")]
    if not versioned:
        print("WARNING: no libcuda.so* found anywhere searched. The build cell below "
              "will automatically retry with -DGGML_CUDA_NO_VMM=ON if the normal "
              "configure fails, which sidesteps this dependency entirely.")
        return
    src = sorted(versioned)[-1]
    target_dir = Path(src).parent
    target = target_dir / "libcuda.so"
    try:
        if not target.exists():
            target.symlink_to(Path(src).name)
        run(["ldconfig"], check=False)
        print(f"OK: created {target} -> {Path(src).name}")
    except PermissionError:
        stub_dir = WORK / "cuda_stub"
        stub_dir.mkdir(exist_ok=True)
        stub = stub_dir / "libcuda.so"
        if not stub.exists():
            stub.symlink_to(src)
        CUDA_STUB_DIR = stub_dir
        print(f"OK (no write access to {target_dir}): created {stub} -> {src}; "
              f"cmake will be told about {stub_dir} via CMAKE_LIBRARY_PATH")

ensure_libcuda_so()


In [ ]:
GGML_CUDA_NO_VMM_USED = False  # recorded for provenance if the fallback below ends up needed

def build(src_dir, targets):
    global GGML_CUDA_NO_VMM_USED
    build_dir = src_dir / "build"
    bin_dir = build_dir / "bin"
    if all((bin_dir / t).exists() for t in targets):
        print(f"[build] {src_dir.name}: {targets} already built, skipping")
        return build_dir

    # Reconfigure from a clean build dir whenever we're actually about to (re)build:
    # CMake caches successfully-resolved toolkit anchor paths (CUDAToolkit_LIBRARY_DIR
    # etc.) across reconfigures of the SAME build dir, so a prior failed attempt's
    # cache could mask a fix that would otherwise now work -- re-running cmake in
    # place is not a reliable retry. (Only reached when the skip check above didn't
    # already return -- an already-succeeded build is never wiped.)
    if build_dir.exists():
        shutil.rmtree(build_dir)

    cmake_env = {"CUDACXX": f"{CUDA_TOOLKIT_ROOT}/bin/nvcc"}
    base_args = ["-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=ON",
                 f"-DCUDAToolkit_ROOT={CUDA_TOOLKIT_ROOT}"]
    if CUDA_STUB_DIR is not None:
        base_args.append(f"-DCMAKE_LIBRARY_PATH={CUDA_STUB_DIR}")

    rc = run(["cmake", "-S", str(src_dir), "-B", str(build_dir), *base_args],
             env=cmake_env, check=False)
    if rc != 0:
        print("[build] primary cmake configure failed -- retrying once with "
              "-DGGML_CUDA_NO_VMM=ON. This disables CUDA virtual memory management "
              "pooling in ggml (a performance/memory-fragmentation feature, not a "
              "correctness one) and, verified directly against ggml's current "
              "CMakeLists.txt, is what gates the target_link_libraries(ggml-cuda "
              "PRIVATE CUDA::cuda_driver) line that's been failing -- so this is a "
              "guaranteed unblock rather than another guess at where libcuda.so "
              "lives, at the cost of that one feature.", flush=True)
        shutil.rmtree(build_dir)
        run(["cmake", "-S", str(src_dir), "-B", str(build_dir),
             *base_args, "-DGGML_CUDA_NO_VMM=ON"], env=cmake_env)
        GGML_CUDA_NO_VMM_USED = True

    run(["cmake", "--build", str(build_dir), "--target", *targets, "-j", str(os.cpu_count())])
    return build_dir

PRISTINE_DIR = WORK / "llama.cpp-pristine"
PRUNE_DIR = WORK / "llama.cpp-prune"

pristine_build = build(PRISTINE_DIR, ["llama-mtmd-cli"])
prune_build = build(PRUNE_DIR, ["llama-mtmd-cli", "llama-server"])

PRISTINE_BIN = pristine_build / "bin" / "llama-mtmd-cli"
PRUNE_BIN = prune_build / "bin" / "llama-mtmd-cli"
PRUNE_SERVER_BIN = prune_build / "bin" / "llama-server"
for p in (PRISTINE_BIN, PRUNE_BIN, PRUNE_SERVER_BIN):
    if not p.exists():
        raise RuntimeError(f"expected build output missing: {p}")
print("build OK:", PRISTINE_BIN, PRUNE_BIN, PRUNE_SERVER_BIN)
if GGML_CUDA_NO_VMM_USED:
    print("NOTE: built with -DGGML_CUDA_NO_VMM=ON (CUDA VMM pooling disabled) -- "
          "this is recorded in the build-desc strings passed to the latency/accuracy "
          "sweeps below, so it's visible in every results JSON's provenance.")


## 7. Verify build provenance

Two-signal check established after the Task 0 build-contamination
incident on CPU (`NOTES.md` "Task 0"): `--version` alone isn't trusted,
`--help | grep visual-keep` presence/absence is cross-checked too.


In [ ]:
def version_of(binpath):
    r = run_capture([str(binpath), "--version"], check=False)
    return r.stderr.strip() or r.stdout.strip()

def has_visual_keep_flag(binpath):
    out = run_capture([str(binpath), "--help"], check=False).stdout
    return "visual-keep" in out

print("--- pristine ---")
print(version_of(PRISTINE_BIN))
if has_visual_keep_flag(PRISTINE_BIN):
    raise RuntimeError("pristine build unexpectedly has --visual-keep")
print("OK: pristine build correctly lacks --visual-keep")

print("--- prune ---")
print(version_of(PRUNE_BIN))
if not has_visual_keep_flag(PRUNE_BIN):
    raise RuntimeError("prune build missing --visual-keep")
print("OK: prune build has --visual-keep")


## 8. Download models and verify checksums

Same files as the CPU/x86 runs (`second-state/Llava-v1.5-7B-GGUF`
Q4_K_M + F16 mmproj) — required for the speed/accuracy numbers to be
comparable across platforms. Skips the download if a correctly-checksummed
file already exists (resumability across a re-run in the same session).


In [ ]:
MODELS_DIR = WORK / "models"
MODELS_DIR.mkdir(exist_ok=True)
MODEL_GGUF = MODELS_DIR / "llava-v1.5-7b-Q4_K_M.gguf"
MMPROJ_GGUF = MODELS_DIR / "llava-v1.5-7b-mmproj-model-f16.gguf"

def ensure_model(path, url, expected_sha256):
    if path.exists() and sha256_file(path) == expected_sha256:
        print(f"OK (cached): {path.name}")
        return
    run(["curl", "-L", "--fail", "--retry", "3", "-o", str(path), url])
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise RuntimeError(f"checksum mismatch for {path.name}: got {actual}, expected {expected_sha256}")
    print(f"OK (downloaded + verified): {path.name}")

ensure_model(MODEL_GGUF,
             "https://huggingface.co/second-state/Llava-v1.5-7B-GGUF/resolve/main/llava-v1.5-7b-Q4_K_M.gguf",
             MODEL_GGUF_SHA256)
ensure_model(MMPROJ_GGUF,
             "https://huggingface.co/second-state/Llava-v1.5-7B-GGUF/resolve/main/llava-v1.5-7b-mmproj-model-f16.gguf",
             MMPROJ_SHA256)


## 9. Verify GPU offload actually happens

A CUDA-compiled binary run without `-ngl` set still defaults to
`n_gpu_layers = -1` = "offload everything" (verified against the actual
compiled struct default on the CPU build, not assumed — see the commit
message for `dump_llamacpp_embd.py`'s `--n-gpu-layers` change). Passing
`-ngl 999` below is therefore belt-and-suspenders, not strictly required
— but the check itself (parsing the model loader's own `"offloaded X/Y
layers to GPU"` log line, source-verified to exist at the pinned commit)
is the actual safety net: if this ever prints 0 offloaded layers, every
downstream number in this notebook would silently be a CPU number
mislabeled as GPU. Hard-fails rather than continuing if that happens.


In [ ]:
import re

VISUAL_KEEP_EXTRA_ARGS = ["-ngl", "999"]

sample_image = WORK / "vision_tokens" / "assets" / "coco_val2017_000000039769.jpg"
probe = run_capture([
    str(PRUNE_BIN), "-m", str(MODEL_GGUF), "--mmproj", str(MMPROJ_GGUF),
    "--image", str(sample_image), "-p", "test", "-n", "1", "--temp", "0",
    *VISUAL_KEEP_EXTRA_ARGS, "-v",
], check=True)
log = probe.stdout + "\n" + probe.stderr
m = re.search(r"offloaded (\d+)/(\d+) layers to GPU", log)
if not m or int(m.group(1)) == 0:
    raise RuntimeError(
        "Did not find evidence of GPU layer offload in the model-load log "
        "(expected a line matching 'offloaded N/M layers to GPU' with N>0). "
        "This means the CUDA build is silently running on CPU. Full log above -- "
        "inspect it, this is the single most important check in the notebook."
    )
print(f"OK: offloaded {m.group(1)}/{m.group(2)} layers to GPU")


## 10. Gate 0.5 — GPU determinism floor

CPU's Gate 1 (`NOTES.md` "Gate 0.5") trusted bitwise `np.array_equal` only
after confirming the CPU vision encode is deterministic run-to-run. GPU
kernels are a different regime — CUDA reduction/atomic ordering is not
always bit-reproducible across runs even on identical inputs. Check this
directly before deciding how to run Gate 1 below, rather than assuming
either regime.


In [ ]:
sys.path.insert(0, str(WORK / "vision_tokens" / "scripts" / "phase1"))

def dump_embd(lib_dir, out_path, image, pruned_abi=False, visual_keep=1.0, visual_prune_method="none",
              n_gpu_layers=999):
    cmd = [sys.executable, str(WORK / "vision_tokens" / "scripts" / "phase1" / "dump_llamacpp_embd.py"),
           "--image", str(image), "--out", str(out_path),
           "--model", str(MODEL_GGUF), "--mmproj", str(MMPROJ_GGUF),
           "--fa", "auto", "--lib-dir", str(lib_dir), "--n-gpu-layers", str(n_gpu_layers)]
    if pruned_abi:
        cmd += ["--pruned-abi", "--visual-keep", str(visual_keep), "--visual-prune-method", visual_prune_method]
    run_capture(cmd, check=True)

import numpy as np
CAT_IMAGE = WORK / "vision_tokens" / "assets" / "phase1" / "cat_336.png"
GATE_DIR = WORK / "gate_checks"
GATE_DIR.mkdir(exist_ok=True)

dump_embd(prune_build / "bin", GATE_DIR / "det_run1.npy", CAT_IMAGE, pruned_abi=True, visual_keep=1.0)
dump_embd(prune_build / "bin", GATE_DIR / "det_run2.npy", CAT_IMAGE, pruned_abi=True, visual_keep=1.0)
a = np.load(GATE_DIR / "det_run1.npy")
b = np.load(GATE_DIR / "det_run2.npy")
GPU_IS_BITWISE_DETERMINISTIC = bool(np.array_equal(a, b))
max_diff = float(np.abs(a - b).max())
print(f"GPU determinism floor: bitwise identical across 2 runs = {GPU_IS_BITWISE_DETERMINISTIC} (max abs diff = {max_diff})")
if not GPU_IS_BITWISE_DETERMINISTIC:
    cos = float(np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b)))
    print(f"Not bit-deterministic on this GPU/build -- mean cosine between the two runs: {cos:.6f}. "
          f"Gate 1 below will use a cosine-similarity criterion instead of np.array_equal, matching "
          f"Phase 1's established floor (0.99 + margin), rather than a bitwise test this backend can't meet.")


## 11. Gate 1 — keep=1.0 vs no-pruning

Pristine build vs prune build @ `--visual-keep 1.0`, both CUDA, both
`-ngl 999`. Criterion depends on what section 10 found: bitwise
`np.array_equal` if the GPU proved deterministic, otherwise a cosine
floor (0.99 + margin — the same threshold Phase 1 already established
for the CPU F16/fp32 numerical floor, reused here rather than invented
fresh). Either way this is insurance before trusting any GPU number
below it, exactly as the x86 run did.


In [ ]:
dump_embd(pristine_build / "bin", GATE_DIR / "pristine_keep1.npy", CAT_IMAGE, pruned_abi=False)
dump_embd(prune_build / "bin", GATE_DIR / "prune_keep1.npy", CAT_IMAGE, pruned_abi=True, visual_keep=1.0, visual_prune_method="cls")

p = np.load(GATE_DIR / "pristine_keep1.npy")
q = np.load(GATE_DIR / "prune_keep1.npy")

if GPU_IS_BITWISE_DETERMINISTIC:
    identical = bool(np.array_equal(p, q))
    print(f"Gate 1 (bitwise): {'PASS' if identical else 'FAIL'} (max abs diff = {float(np.abs(p-q).max())})")
    if not identical:
        raise RuntimeError("Gate 1 FAILED (bitwise) -- see printed diff above. Stopping before any downstream "
                            "GPU number is generated against a build that may not match the CPU-validated one.")
else:
    cos = float(np.dot(p.flatten(), q.flatten()) / (np.linalg.norm(p) * np.linalg.norm(q)))
    GATE1_COSINE_FLOOR = 0.99
    passed = cos >= GATE1_COSINE_FLOOR
    print(f"Gate 1 (cosine, GPU not bit-deterministic): mean cosine = {cos:.6f}, floor = {GATE1_COSINE_FLOOR} "
          f"-> {'PASS' if passed else 'FAIL'}")
    if not passed:
        raise RuntimeError(f"Gate 1 FAILED (cosine {cos:.6f} < {GATE1_COSINE_FLOOR}) -- stopping.")
print("Gate 1: PASS")


## 12. Latency sweep

Reuses `sweep_prune.py`/`bench_baseline.py` exactly as the CPU/x86 runs
did — same resumable per-cell driver, cooldown and platform tag from the
config cell. `sweep_prune.py` previously had no way to forward an extra
CLI flag (only its own hardcoded `--visual-keep`/`--visual-prune-method`)
— added a generic `--extra-arg` passthrough (same pattern as
`bench_baseline.py` already had) so `-ngl 999` actually reaches the
binary here, rather than only appearing in the build-desc string below
without being a real argv entry.


In [ ]:
VT = WORK / "vision_tokens"
BUILD_DESC = (f"cmake -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON "
              f"{'-DGGML_CUDA_NO_VMM=ON ' if GGML_CUDA_NO_VMM_USED else ''}"
              f"(offloaded, -ngl 999, {gpu_name})")
run([
    sys.executable, str(VT / "scripts" / "sweep_prune.py"),
    "--bin", str(PRUNE_BIN), "--llama-repo", str(PRUNE_DIR),
    "--ratios", LATENCY_RATIOS, "--runs", str(LATENCY_RUNS), "--warmup", str(LATENCY_WARMUP),
    "--cooldown-s", str(LATENCY_COOLDOWN_S),
    "--tag-prefix", "p2_sweep_kaggle_gpu",
    "--platform-tag", PLATFORM_TAG,
    "--build-desc", BUILD_DESC,
    "--extra-arg", "-ngl", "--extra-arg", "999",
], cwd=VT, env={"LLAMA_CPP_DIR": str(PRUNE_DIR)})


## 13. Latency analysis

Same `sweep_analysis.py` used for CPU/x86 — fits the prune-overhead
model, computes the H1 fraction-of-theoretical-ceiling number, which is
this notebook's central comparative output (CPU vs GPU on that specific
quantity).


In [ ]:
run([sys.executable, str(VT / "scripts" / "sweep_analysis.py"), "--tag-prefix", "p2_sweep_kaggle_gpu"], cwd=VT)

analysis = json.loads((VT / "results" / "p2_sweep_kaggle_gpu_analysis.json").read_text())
print(f"{'keep':>7} {'K':>4} {'ttft_llm_ms':>14} {'speedup':>10} {'frac_ceil_h1':>13}")
for row in analysis["table"]:
    fc = row["fraction_of_theoretical_ceiling_h1"]
    print(f"{row['keep']:>7.3f} {row['K_image_tokens']:>4d} {row['ttft_llm_ms_mean']:>10.1f}±{row['ttft_llm_ms_std']:>3.0f} "
          f"{row['speedup_ttft_llm']:>9.2f}x {f'{fc*100:.1f}%' if fc is not None else 'n/a':>13}")


## 14. Materialize the 200 pinned TextVQA images

The manifest (`assets/phase1/textvqa200_manifest.jsonl`) only records
metadata; images are re-derived by streaming the dataset and
cross-checking question text at each index against the pin (fails loudly
on any drift — see the script's docstring).


In [ ]:
TEXTVQA_DATA = WORK / "textvqa_data"
run([sys.executable, str(VT / "scripts" / "phase1" / "materialize_textvqa_images.py"), "--out", str(TEXTVQA_DATA)])


## 15. Accuracy sweep — TextVQA vs keep ratio

See `textvqa_keep_sweep.py`'s docstring for the mode-selection logic:
tries server mode (one `llama-server` per ratio, ~200x fewer model loads
than CLI mode) only after an empirical probe confirms the server's
chat-template rendering produces identical text to the already-validated
CLI path on one sample. Any mismatch, or if the server doesn't come up,
falls back to CLI mode for the whole sweep — watch the printed output
from this cell, it says explicitly which mode was used and why.

Resumable per-sample: a disconnect mid-ratio does not lose completed
samples on re-run.


In [ ]:
run([
    sys.executable, str(VT / "scripts" / "phase1" / "textvqa_keep_sweep.py"),
    "--data", str(TEXTVQA_DATA),
    "--bin", str(PRUNE_BIN), "--server-bin", str(PRUNE_SERVER_BIN), "--llama-repo", str(PRUNE_DIR),
    "--model", str(MODEL_GGUF), "--mmproj", str(MMPROJ_GGUF),
    "--ratios", ACCURACY_RATIOS, "--limit", str(ACCURACY_LIMIT),
    "--tag-prefix", "p3_textvqa_kaggle_gpu",
    "--platform-tag", PLATFORM_TAG,
    "--build-desc", BUILD_DESC,
    "--extra-arg", "-ngl", "--extra-arg", "999",
    "--mode", ACCURACY_MODE,
], cwd=VT, env={"LLAMA_CPP_DIR": str(PRUNE_DIR)})


## 16. Accuracy summary


In [ ]:
summaries = sorted(glob.glob(str(VT / "results" / "*_p3_textvqa_kaggle_gpu_keep*_summary.json")))
print(f"{'keep':>7} {'n_scored':>9} {'n_failed':>9} {'acc_mean':>10} {'acc_std':>9} {'mode':>8}")
for f in summaries:
    s = json.loads(Path(f).read_text())
    acc = s["acc_mean"]
    print(f"{s['visual_keep']:>7.3f} {s['n_scored']:>9d} {s['n_failed']:>9d} "
          f"{f'{acc*100:.1f}%' if acc is not None else 'n/a':>10} "
          f"{s['acc_std']*100 if s['acc_std'] is not None else float('nan'):>8.1f}% {s['mode']:>8}")


## 17. MME / POPE — not implemented (disabled)

Out of scope for this notebook as written. MME's scoring (accuracy +
"accuracy+", the latter requiring both images in a yes/no pair correct)
and POPE's (precision/recall/F1 on a yes/no-imbalanced hallucination
probe) are each their own metric definitions, distinct from TextVQA's
soft-VQA accuracy — building correct harnesses for both is a real,
separate piece of work (dataset loading + a purpose-built scorer for
each), not something to bolt on quickly inside an already-large notebook
without the same care the TextVQA harness got.

To add either: follow the `textvqa_sim.py` -> `textvqa_llamacpp.py` ->
`textvqa_keep_sweep.py` pattern (pin a manifest, validate the scoring
code against a known reference, then wire it into a keep-ratio sweep
script) and add a call to it below. Left as an explicit stub rather than
a silent omission.


In [ ]:
RUN_MME = False
RUN_POPE = False
if RUN_MME or RUN_POPE:
    raise NotImplementedError("MME/POPE harnesses are not implemented in this notebook -- see the markdown above")
print("MME/POPE: skipped (not implemented, see markdown above)")


## 18. Output files

Everything below is under `/kaggle/working/vision_tokens/results/` —
that's what to grab from Kaggle's **Output** panel after the run (or
"Save & Run All" to persist it as a Version).


In [ ]:
result_files = sorted((VT / "results").glob("*.json")) + sorted((VT / "results" / "raw").rglob("*"))
print(f"{len(result_files)} files under vision_tokens/results/:\n")
for f in result_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(VT)}  ({size_kb:.1f} KiB)")

print(f"\nAlso: {GATE_DIR.relative_to(WORK)}/ (Gate 0.5 + Gate 1 raw .npy embeddings, for independent inspection)")


## Done

Files live under `/kaggle/working/vision_tokens/results/` (note: nested
under `vision_tokens/`, not directly under `/kaggle/working/results/` —
section 18 above and this notebook's README both point at the real path).
Pull `*.json` and `raw/` from Kaggle's Output panel into this repo's own
`results/` directory locally.

The **latency** sweep's output (`p2_sweep_kaggle_gpu_analysis.json`) is
produced by the exact same `sweep_analysis.py` the CPU/x86 runs use, so
it's already the identical JSON shape, tagged with this run's
`platform_tag` (printed in section 2) — no special-casing needed.

The **accuracy** sweep's per-ratio summary JSONs are a new shape —
`textvqa_keep_sweep.py` is new code, there was no prior keep-ratio
accuracy schema on CPU/x86 to match against. Each summary records
`visual_keep`, `platform_tag`, `mode` (`server`/`cli`), `acc_mean`,
`acc_std`, `n_scored`/`n_failed`, and full build provenance — self-
describing, but don't assume it slots into existing tooling without
checking field names first.
